<a href="https://colab.research.google.com/github/Nawaf-Rayhan585/YOLO_Projects/blob/main/license-plate-recognition/train_plate_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# License Plate Detector — Train on Colab T4, Use Anywhere

Trains a YOLOv8 model to *locate* license plates in a frame. The actual
text reading is handled separately by EasyOCR in `anpr_inference.py` — this
notebook only needs to learn "where is the plate", not what's written on it.

**How to use:**
1. Runtime -> Change runtime type -> GPU -> T4
2. Run cells top to bottom
3. Download `best.pt` and drop it into this folder

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Install packages

In [ ]:
!pip install ultralytics roboflow -q

## 3. Get a license plate dataset

Search "license plate detection" on [Roboflow Universe](https://universe.roboflow.com),
download in **YOLOv8** format, and paste your snippet's values below.

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = "YOUR_API_KEY"
WORKSPACE = "YOUR_WORKSPACE"
PROJECT = "YOUR_PROJECT"
VERSION = 1

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8")

print(dataset.location)

## 4. Train

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    device=0,
    project="plate_detector",
    name="train1",
)

## 5. Validate

In [ ]:
metrics = model.val()
print(metrics)

## 6. Download your trained model

In [ ]:
from google.colab import files

files.download("plate_detector/train1/weights/best.pt")

## 7. Use it locally

```bash
pip install -r requirements.txt
python anpr_inference.py --model best.pt --source your_video.mp4
```

`anpr_inference.py` runs this detector then feeds each detected plate crop
through EasyOCR to read the actual characters — no extra training needed
for the OCR step.